Será feito o tratamento dos dados removendo todos os registros anteriores a 1990. Os dados desses anos são muito imprecisos, mal formatados ou inexistentes (incompletos).

O races.csv é a chave de tudo, porque todos os outros CSVs se conectam a ele via raceId. Então a estratégia é:
Filtrar o races.csv primeiro, e deixar o merge fazer o resto automaticamente. Depois do filtro o races só tem corridas de 1990+, então quando você faz o merge com results, lap_times e pit_stops pelo raceId, registros de anos anteriores simplesmente não encontram correspondência e são descartados automaticamente.

In [1]:
import pandas as pd

# o dataset do Ergast/Jolpica usa "\N" como marcador de ausência
na = ["\\N"]

races = pd.read_csv("../databases/races.csv", na_values=na)
results = pd.read_csv("../databases/results.csv", na_values=na)
lap_times = pd.read_csv("../databases/lap_times.csv", na_values=na)
pit_stops = pd.read_csv("../databases/pit_stops.csv", na_values=na)
drivers = pd.read_csv("../databases/drivers.csv", na_values=na)
constructors = pd.read_csv("../databases/constructors.csv", na_values=na)

# filtra o races para 1990 em diante
races = races[races["year"] >= 1990]

# a partir daqui, todos os merges já herdam o filtro automaticamente
# pois só existem raceIds de 1990+ no races filtrado
df = results.merge(races, on="raceId")
df = df.merge(drivers, on="driverId")
df = df.merge(constructors, on="constructorId")
df_laps = lap_times.merge(races[["raceId", "year", "name"]], on="raceId")
df_pits = pit_stops.merge(races[["raceId", "year", "name"]], on="raceId")

print(f"DataFrame principal: {df.shape}")
print(f"Lap times filtrado:  {df_laps.shape}")
print(f"Pit stops filtrado:  {df_pits.shape}")


DataFrame principal: (14329, 47)
Lap times filtrado:  (589081, 8)
Pit stops filtrado:  (11371, 9)


Isso vai revelar o que precisa ser tratado colunas com muitos nulos, datas em formato errado, tipos incorretos e a partir daí a gente define o que fazer em cada caso.

In [2]:
# ver quantos valores ausentes existem em cada coluna
print(df.isnull().sum())

# ver os tipos de cada coluna
print(df.dtypes)

# ver uma amostra
print(df.head())

resultId               0
raceId                 0
driverId               0
constructorId          0
number_x               0
grid                   0
position            4319
positionText           0
positionOrder          0
points                 0
laps                   0
time_x              8777
milliseconds        8777
fastestLap          6077
rank                5819
fastestLapTime      6077
fastestLapSpeed     6077
statusId               0
year                   0
round                  0
circuitId              0
name_x                 0
date                   0
time_y              6039
url_x                  0
fp1_date           12530
fp1_time           12970
fp2_date           12530
fp2_time           12970
fp3_date           12890
fp3_time           13270
quali_date         12530
quali_time         12970
sprint_date        13969
sprint_time        14029
driverRef              0
number_y            7768
code                4170
forename               0
surname                0


## Tratamento dos valores ausentes

Os nulos do dataset têm causas semânticas distintas, não há regra única. A estratégia por grupo:

1. **Colunas de sessões não usadas** (`fp1_*`, `fp2_*`, `fp3_*`, `quali_*`, `sprint_*`) e `url_*`: dropadas, pois >85% dos registros são nulos (dados só existem para corridas recentes) e não serão usadas na análise.
2. **`number_y`** (número permanente do piloto, só existe a partir de 2014): dropada — usaremos `number_x` (número do carro na corrida) renomeada para `number`.
3. **`code`** (sigla de 3 letras): preenchida com `surname[:3].upper()` como fallback para pilotos antigos.
4. **`position`, `time_x`, `milliseconds`**: nulos *informativos* (DNF/abandono/desclassificação). Mantidos como `NaN` — `positionText` e `statusId` já indicam o motivo. Análises de tempo/posição final devem filtrar `position.notna()`.
5. **`fastestLap*` e `rank`**: dado histórico não coletado antes de 2004. Mantidos como `NaN`; análises desses campos devem restringir a `year >= 2004`.
6. **`time_y`** (horário UTC da corrida): dropada, não será usada.

In [3]:
# colunas a remover do df principal
cols_drop = [
    "fp1_date", "fp1_time", "fp2_date", "fp2_time", "fp3_date", "fp3_time",
    "quali_date", "quali_time", "sprint_date", "sprint_time",
    "url_x", "url_y", "url",
    "number_y", "time_y",
]
df = df.drop(columns=cols_drop)

# number_x -> number (número do carro na corrida)
df = df.rename(columns={"number_x": "number", "name_x": "race_name", "name_y": "constructor_name"})

# code: fallback com as 3 primeiras letras do sobrenome em maiúsculo
df["code"] = df["code"].fillna(df["surname"].str[:3].str.upper())

# confere o resultado
print(f"Shape final: {df.shape}")
print("\nNulos remanescentes (esperados — DNFs e fastest lap pré-2004):")
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape final: (14329, 32)

Nulos remanescentes (esperados — DNFs e fastest lap pré-2004):
position           4319
time_x             8777
milliseconds       8777
fastestLap         6077
rank               5819
fastestLapTime     6077
fastestLapSpeed    6077
dtype: int64


## Resumo geral da base (KPIs para Dashboard 1)

Indicadores agregados que vão alimentar os cards do painel executivo.

In [4]:
kpis = {
    "Temporadas":        df["year"].nunique(),
    "Corridas":          df["raceId"].nunique(),
    "Pilotos":           df["driverId"].nunique(),
    "Equipes":           df["constructorId"].nunique(),
    "Circuitos":         df["circuitId"].nunique(),
    "Nacionalidades":    df["nationality_x"].nunique(),
    "Período":           f'{df["year"].min()} a {df["year"].max()}',
    "Vencedor recordista":      df[df["position"] == 1]["surname"].value_counts().idxmax(),
    "Equipe recordista":        df[df["position"] == 1]["constructor_name"].value_counts().idxmax(),
}

print("=" * 50)
print("RESUMO DA BASE — Fórmula 1 (1990 — atual)")
print("=" * 50)
for k, v in kpis.items():
    print(f"{k:<22}: {v}")

RESUMO DA BASE — Fórmula 1 (1990 — atual)
Temporadas            : 35
Corridas              : 641
Pilotos               : 209
Equipes               : 61
Circuitos             : 46
Nacionalidades        : 35
Período               : 1990 a 2024
Vencedor recordista   : Hamilton
Equipe recordista     : Ferrari


## Vitórias por equipe ao longo dos anos

Para **cada equipe** que venceu ao menos uma corrida no período, gera um gráfico de linha individual com a contagem de vitórias por temporada. Anos sem vitória aparecem como zero (eixo contínuo), permitindo enxergar **eras de dominância** e **declínios** isoladamente por equipe.

In [12]:
import plotly.express as px

# só vitórias
vitorias = df[df["position"] == 1]

# equipes ordenadas por total de vitórias (decrescente)
equipes = vitorias["constructor_name"].value_counts().index.tolist()

# range de anos para preencher zeros
anos = list(range(df["year"].min(), df["year"].max() + 1))

# guarda as figuras para uso posterior (Dash, export HTML, etc.)
figs = {}

for equipe in equipes:
    # contagem de vitórias por ano da equipe
    contagem = (
        vitorias[vitorias["constructor_name"] == equipe]
        .groupby("year")
        .size()
        .reindex(anos, fill_value=0)
        .reset_index(name="vitorias")
        .rename(columns={"index": "year"})
    )

    total = int(contagem["vitorias"].sum())

    fig = px.line(
        contagem,
        x="year",
        y="vitorias",
        markers=True,
        title=f"{equipe} — vitórias por temporada (total: {total})",
        labels={"year": "Temporada", "vitorias": "Corridas vencidas"},
    )
    fig.update_layout(
        template="plotly_white",
        width=900, height=380,
        showlegend=False,
    )
    fig.update_traces(line=dict(width=2))
    fig.update_xaxes(dtick=2)
    fig.update_yaxes(dtick=max(1, contagem["vitorias"].max() // 5), rangemode="tozero")

    figs[equipe] = fig
    fig.show()